## Get DO model progress and control solve with watsonx.ai Runtime

This notebook shows you how to deploy a DOcplex model, publish your own progress kpis and take actions on the progress speed using the watsonx.ai Python Client.

This notebook runs on Python.

**Table of contents:**

- [Set up the watsonx.ai client](#setup)
- [Create a client instance](#create)
- [Prepare your model archive](#prepare)
- [Upload your model on watsonx.ai Runtime](#upload)
- [Create a deployment](#deploy)
- [Create and monitor a job for your deployed model](#job)
- [Summary](#summary)

<a id='setup'></a>
### Set up the watsonx.ai client

Before you use the sample code in this notebook, you must:

- create a <a href="https://cloud.ibm.com/catalog?category=ai" target="_blank" rel="noopener noreferrer">watsonx.ai Runtime Service</a> instance. A free plan is offered and information about how to create the instance can be found at <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/ml-overview.html?context=cpdaas" target="_blank" rel="noopener noreferrer"> https://dataplatform.cloud.ibm.com/docs/content/wsj/analyze-data/ml-overview.html?context=cpdaas.</a>


Import the watsonx.ai client library.

In [ ]:
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai import Credentials

<a id='create'></a>
### Create a client instance

Use your IBM Cloud API key. You can find information on how to get your API key <a href="https://dataplatform.cloud.ibm.com/docs/content/DO/WML_Deployment/DeployModelRest.html?audience=wdp&context=cpdaas#tasktask_deploymodelREST__prereq_el2_nft_bhb">here</a> and the instance URL <a href="https://cloud.ibm.com/apidocs/machine-learning#endpoint-url">here</a>.

In [ ]:
# Instantiate a client using credentials
credentials = Credentials(
      api_key = "<API_key>",
      url = "<instance_url>"
)

client = APIClient(credentials)

In [ ]:
client.version

<a id='prepare'></a>
### Prepare your model archive

Use the `write_file` command to write the model to a `model.py` file.

This model is a simple geometric problem that is easily scalable and thus used to demonstrate the engine progress. Any other scalable Mixed Integer Programming model would also work.

The model solves this geometric puzzle:  
Start with a pattern of circles placed in rows piled one on top of the other with decreasing numbers in each row to form a triangle. With N circles at the bottom, then N-1 circles in the adjacent row, then N-2 in the next row etc ... until there is just 1 circle in the top row, you must decide which circles to color in, so that the maximum number of circles are colored in. There is however the constraint that no 3 selected circles form a triangle. Hence the center of any circle is considered as a vertex for a potential triangle.

Use the `tar` command to create a tar archive.

In [ ]:
%mkdir model

In [ ]:
%%writefile model.py

from docplex.mp.model import Model
from docplex.mp.progress import ProgressListener
from docplex.util.environment import get_environment

# You use a standard simple listener to track progress info and publish it.
# You could use another more complex listener to follow the engine, its internal progress, ...
# To keep it simple: you will publish the gap and best bound and keep a history.
class MyProgressListener(ProgressListener):
    def notify_progress(self, progress_data):
        bound = progress_data.best_bound
        gap = progress_data.mip_gap
                
        sd = {'MY_BEST_BOUND': bound,'MY_MIP_GAP': gap}
        
        #print(sd)
        get_environment().update_solve_details(sd)



def build_hearts(r, **kwargs):
    # initialize the model
    mdl = Model('love_hearts_%d' % r, **kwargs)
    mdl.parameters.timelimit = 180
    mdl.context.solver.auto_publish = False
    mdl.add_progress_listener(MyProgressListener())

    # the dictionary of decision variables, one variable
    # for each circle with i in (1 .. r) as the row and
    # j in (1 .. i) as the position within the row    
    idx = [(i, j) for i in range(1, r + 1) for j in range(1, i + 1)]
    a = mdl.binary_var_dict(idx, name=lambda ij: "a_%d_%d" % ij)

    # the constraints - enumerate all equilateral triangles
    # and prevent any such triangles from being chosen by keeping
    # the number of chosen circles with adjacent vertices below 3

    # for each row except the last
    for i in range(1, r):
        # for each position in this row
        for j in range(1, i + 1):
            # for each triangle of side length (k) with its upper vertex at
            # (i, j) and its sides parallel to those of the overall shape
            for k in range(1, r - i + 1):
                # the sets of 3 points at the same distances clockwise along the
                # sides of these triangles form k equilateral triangles
                for m in range(k):
                    u, v, w = (i + m, j), (i + k, j + m), (i + k - m, j + k - m)
                    mdl.add(a[u] + a[v] + a[w] <= 2)

    mdl.maximize(mdl.sum(a))
    return mdl

from docplex.util.environment import get_environment
       

mdl = build_hearts(11)
mdl.solve(log_output=False)

In [ ]:
import tarfile
def reset(tarinfo):
    tarinfo.uid = tarinfo.gid = 0
    tarinfo.uname = tarinfo.gname = "root"
    return tarinfo
tar = tarfile.open("model.tar.gz", "w:gz")
tar.add("model.py", arcname="model.py", filter=reset)
tar.close()

<a id='upload'></a>
### Upload your model on watsonx.ai Runtime

Store model in watsonx.ai Runtime with:
* the tar archive previously created,
* metadata including the model type and runtime

Get the `model_uid`.

In [ ]:
# Find the space ID

space_name = "<space_name>"

space_id = [x['metadata']['id'] for x in client.spaces.get_details()['resources'] if x['entity']['name'] == space_name][0]

client = APIClient(credentials, space_id = space_id)

In [ ]:
mnist_metadata = {
    client.repository.ModelMetaNames.NAME: "MyModel",
    client.repository.ModelMetaNames.DESCRIPTION: "Model for Loving Hearts",
    client.repository.ModelMetaNames.TYPE: "do-docplex_22.1",
    client.repository.ModelMetaNames.SOFTWARE_SPEC_ID: client.software_specifications.get_id_by_name("do_22.1"),
}

model_details = client.repository.store_model(model='/home/wsuser/work/model.tar.gz', meta_props=mnist_metadata)
#model='/home/wsuser/work/model.tar.gz', 
model_uid = client.repository.get_model_id(model_details)

<a id='deploy'></a>
### Create a deployment 

Create a batch deployment for the model, providing information such as:
* the maximum number of compute nodes
* the T-shirt size of the compute nodes

Get the `deployment_uid`.

In [ ]:
meta_props = {
    client.deployments.ConfigurationMetaNames.NAME: "Loving Hearts Deployment",
    client.deployments.ConfigurationMetaNames.DESCRIPTION: "Loving Hearts Deployment",
    client.deployments.ConfigurationMetaNames.BATCH: {},
    client.deployments.ConfigurationMetaNames.HARDWARE_SPEC: {'name': 'S', 'num_nodes': 1}
}

deployment_details = client.deployments.create(model_uid, meta_props=meta_props)

deployment_uid = client.deployments.get_id(deployment_details)

# print deployment id if needed
# print( deployment_uid )

In [ ]:
# List all existing deployments

client.deployments.list()

<a id='job'></a>
### Create and monitor a job  for your deployed model

Create a payload containing inline input data.

Create a new job with this payload and the deployment.
No specific parameter.

Get the `job_uid`.

In [ ]:
solve_payload = {
    "solve_parameters" : {
        "oaas.logAttachmentName":"log.txt",
        "oaas.logTailEnabled":"true",
        "oaas.resultsFormat": "XML"
    },
    client.deployments.DecisionOptimizationMetaNames.INPUT_DATA: [
    ],
    client.deployments.DecisionOptimizationMetaNames.OUTPUT_DATA: [
        {
            "id":".*\\.xml"
        },
        {
            "id":"log.txt"
        }
    ]
}
job_details = client.deployments.create_job(deployment_uid, solve_payload)
job_uid = client.deployments.get_job_id(job_details)

Display job status until it is completed.

The first job of a new deployment might take some time as a compute node must be started.

Follow the CPLEX progress, and stop the job if the engine does not converge fast enough.

In [ ]:
from time import sleep

manual_stop = False
gap_history = None
bestobj_history = None
no_progress = 0
while no_progress < 10 and manual_stop is False and job_details['entity']['decision_optimization']['status']['state'] not in ['completed', 'failed', 'canceled']:
    state = job_details['entity']['decision_optimization']['status']['state']
    if state == "running":
        details = job_details['entity']['decision_optimization']['solve_state']['details']
        if 'MY_MIP_GAP' in details:
            gap = float(details['MY_MIP_GAP'])
            bestobj = float(details['MY_BEST_BOUND'])

            msg = "gap = {0}%, best obj = {1}".format(gap*100, bestobj)
            print(msg)

            if gap_history is not None:
                if float(gap) == float(gap_history):
                    print("No progress: {0}/{1}% gap and {2}/{3} bestobj".format(gap_history, gap, bestobj_history, bestobj))
                    no_progress+=1
                elif 100*(float(gap_history) - float(gap)) <= 0.1:
                    manual_stop = True
                    print("Stopping as the gap is not converging fast enough: {0}/{1}% gap and {2}/{3} bestobj".format(gap_history, gap, bestobj_history, bestobj))
                    no_progress = 0
                else:
                    print("converging well with {0}".format(float(gap_history) - float(gap)))
                    no_progress = 0
            bestobj_history = bestobj
            gap_history = gap
        else:
            print(state)
    else:
        print(state)
    sleep(5)
    job_details=client.deployments.get_job_details(job_uid)
if manual_stop is False:
    if no_progress < 10:
        print("No manual stop was triggered")
    else:
        print("Stopping the solve as the progress is too slow")
        client.deployments.delete_job(job_uid)
else:
    client.deployments.delete_job(job_uid)
#print(job_details['entity']['decision_optimization']['solve_state']['solve_status'])

In [ ]:
print(job_details)

### Delete the deployment

Use the following method to delete the deployment.

In [ ]:
client.deployments.delete(deployment_uid)

<a id='summary'></a>
### Summary and next steps

You've successfully completed this notebook! 

You've learned how to:

- work with the watsonx.ai client
- prepare your model archive and upload your model on watsonx.ai Runtime
- create a deployment
- create and monitor a job with inline data for your deployed model

Check out our online documentation for more samples, tutorials and documentation:
* <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/welcome-main.html?context=cpdaas" target="_blank" rel="noopener noreferrer">IBM Cloud Pak for Data as a Service documentation</a>
* <a href="https://dataplatform.cloud.ibm.com/docs/content/wsj/getting-started/welcome-main.html?context=wx" target="_blank" rel="noopener noreferrer">IBM watsonx.ai documentation</a>

<hr>
Copyright © 2019-2026. This notebook and its source code are released under the terms of the MIT License.